# Session XII: Working with APIs and JSON Data

> Requires internet connection. All APIs are free, no auth needed.

---

## Exercise 1: First API Call

Fetch exchange rates from `https://open.er-api.com/v6/latest/USD`. Print the status code, the base currency (the `base_code` field) and 5 rates (inside `rates`). Handle errors.

### How to approach this code

The request demonstrates the read cycle: send GET, inspect status, parse JSON, and display selected rates. A timeout bounds waiting and `raise_for_status()` exposes HTTP failures. The keys used to access the response must match the actual API schema.


In [6]:
import requests

# TODO: GET request, check status, parse JSON, print 5 rates, try/except
url = "https://open.er-api.com/v6/latest/USD"
response = requests.get(url, timeout=10)
response.raise_for_status()
data = response.json()

print(response.status_code) 
print(response.text) 
print(data["base_code"])   
print(data["rates"]["EUR"])

200
{"result":"success","provider":"https://www.exchangerate-api.com","documentation":"https://www.exchangerate-api.com/docs/free","terms_of_use":"https://www.exchangerate-api.com/terms","time_last_update_unix":1795651351,"time_last_update_utc":"Thu, 26 Nov 2026 00:02:31 +0000","time_next_update_unix":1795739181,"time_next_update_utc":"Fri, 27 Nov 2026 00:26:21 +0000","time_eol_unix":0,"base_code":"USD","rates":{"USD":1,"AED":3.6725,"AFN":66.198739,"ALL":82.076868,"AMD":360.058557,"ANG":1.79,"AOA":926.362437,"ARS":1608.6204,"AUD":1.513274,"AWG":1.79,"AZN":1.699249,"BAM":1.783895,"BBD":2,"BDT":125.182224,"BGN":1.783895,"BHD":0.376,"BIF":3008.303573,"BMD":1,"BND":1.290916,"BOB":11.781791,"BRL":5.884106,"BSD":1,"BTN":92.337014,"BWP":14.521022,"BYN":3.006007,"BZD":2,"CAD":1.422849,"CDF":2303.216493,"CHF":0.831365,"CLF":0.026138,"CLP":1033.121916,"CNH":6.671089,"CNY":6.683199,"COP":3604.230999,"CRC":443.966094,"CUP":24,"CVE":100.571693,"CZK":22.290068,"DJF":177.721,"DKK":6.834212,"DOP":59.8

---
## Exercise 2: Currency Converter

Build `convert_currency(amount, from_curr, to_curr)` with error handling and timeout. Get the rates for the source currency from `https://open.er-api.com/v6/latest/{FROM}` (for example `https://open.er-api.com/v6/latest/EUR`).

> This API answers an unknown currency code with status 200 and the body `{"result": "error", "error-type": "unsupported-code"}`, so also check that `result` is `"success"` before reading `rates`.

### How to approach this code

Currency conversion retrieves rates relative to the source currency and multiplies the amount by the target rate. Normalize currency codes and handle missing rates or request failures. If `None` represents failure, callers must check it before using the result in arithmetic or formatting.


In [15]:
import requests

def convert_currency(amount: float, from_curr: str, to_curr: str) -> float | None:
    from_curr = from_curr.upper()
    to_curr = to_curr.upper()
    try:
        url = f"https://open.er-api.com/v6/latest/{from_curr}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("result") != "success":
            print(f"Unsupported source currency: {from_curr}")
            return None

        rate = data["rates"].get(to_curr)
        if rate is None:
            print(f"Unsupported target currency: {to_curr}")
            return None

        return amount * rate
    except requests.exceptions.Timeout:
        print("The server did not respond in time.")
    except requests.exceptions.HTTPError as exc:
        print(f"HTTP error: {exc.response.status_code}")
    except requests.exceptions.ConnectionError:
        print("Could not connect: check the network and the host name.")
    except requests.exceptions.RequestException as exc:
        print(f"Request failed: {exc}")
    return None


print(convert_currency(100, "USD", "EUR"))
print(convert_currency(50, "EUR", "GBP"))
print(convert_currency(100, "USD", "INVALID"))

91.21000000000001
41.85685
Unsupported target currency: INVALID
None


---
## Exercise 3: Country Data Fetcher

Download the course country dataset from `https://raw.githubusercontent.com/ie-tech-university/programming-with-python-i/main/data/countries.json` (a JSON list in the REST Countries format: `name.common`, `capital`, `population`, `region`, …) and build `fetch_country(name)`. It returns a dict with name, capital, population and region for the country whose `name.common` matches `name` (case-insensitive), or `None` if there is no match. Handle HTTP errors such as a 404 or a timeout.

> The public REST Countries API v3.1 was shut down in 2026 (v5 needs an API key), so the course serves the same data from its repository. Offline? Read `data/countries.json` from your clone instead.

### How to approach this code

The country fetcher converts an API response into a small dictionary for the rest of the program. Search responses may be lists with nested fields, so the code must select a result and extract fields carefully. A nonexistent-country test demonstrates handling a 404 as an expected failure case.


In [21]:
import requests

def fetch_country(name: str) -> dict | None:
    try:
        url = "https://raw.githubusercontent.com/ie-tech-university/programming-with-python-i/main/data/countries.json"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        fetched_data =  response.json()

        for data in fetched_data:
            if data.get("name").get("common").lower() == name:
                return data
        return f"Country {name} not found!"
    except requests.exceptions.Timeout:
        print("The server did not respond in time.")
    except requests.exceptions.HTTPError as exc:
        print(f"HTTP error: {exc.response.status_code}")
    except requests.exceptions.ConnectionError:
        print("Could not connect: check the network and the host name.")
    except requests.exceptions.RequestException as exc:
        print(f"Request failed: {exc}")
    return None

print(fetch_country("spain"))
print(fetch_country("atlantis"))

{'name': {'common': 'Spain', 'official': 'Kingdom of Spain'}, 'cca2': 'ES', 'cca3': 'ESP', 'capital': ['Madrid'], 'region': 'Europe', 'subregion': 'Southern Europe', 'languages': {'spa': 'Spanish'}, 'currencies': {'EUR': {'name': 'Euro', 'symbol': '€'}}, 'area': 505992, 'population': 49355143, 'independent': True, 'unMember': True, 'flag': '🇪🇸'}
Country atlantis not found!


---
## Exercise 4: API Caching

Build `cached_fetch(url, cache_file, max_age=3600)` that saves responses to file and reuses them while they are fresh. Store the URL together with the data, so a cache file written for one URL is never returned for a different URL.

### How to approach this code

The cache compares the saved file's age to `max_age`; fresh JSON is reused, while stale or absent data triggers a request and a new save. This reduces network traffic. A reliable implementation distinguishes malformed cache data from a network failure and does not overwrite good cached data after a failed request.


In [ ]:
import json
import os
import time
import requests

def cached_fetch(url, cache_file, max_age=3600):
    if os.path.exists(cache_file):
        age = time.time() - os.path.getmtime(cache_file)
        if age < max_age:
            try:
                with open(cache_file, encoding="utf-8") as f:
                    cached = json.load(f)
                if cached.get("url") == url:
                    return cached["data"]
            except (json.JSONDecodeError, KeyError):
                print("Cache file is corrupt, refetching.")

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()       
    except requests.exceptions.RequestException as exc:
        print(f"Request failed: {exc}")
        data = None

    if data is not None:              
        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump({"url": url, "data": data}, f, indent=2)
    return data
base = "https://open.er-api.com/v6/latest/"
usd = cached_fetch(base + "USD", "api_cache.json")
eur = cached_fetch(base + "EUR", "api_cache.json")
print(usd["base_code"], eur["base_code"])  # USD EUR


data = cached_fetch("https://open.er-api.com/v6/latest/EUR", "api_cache.json", 60)
if data: print(f"EUR->USD: {data.get('rates', {}).get('USD')}")
# Second call should use the cache
data2 = cached_fetch("https://open.er-api.com/v6/latest/EUR", "api_cache.json", 60)
# A different URL must not reuse the EUR data
data3 = cached_fetch("https://open.er-api.com/v6/latest/USD", "api_cache.json", 60)
if data3: print(data3.get("base_code"))  # USD

Cache file is corrupt, refetching.
USD EUR
EUR->USD: 1.096373
USD


---
## Exercise 5: Multi-Currency Report

**Capstone.** Build `multi_currency_report(transactions)` that groups expenses by original currency and shows EUR totals.

### How to approach this code

Transactions include source-currency values and normalized EUR values. Grouping by source currency is valid, but summing amounts from different currencies together is not; use EUR for a common comparison. This example connects API rates with grouping and report calculations.


In [31]:
# Given sample data:
transactions = [
    {"amount_eur": 25.0, "original_amount": 27.5, "original_currency": "USD", "category": "Food", "description": "Lunch in NYC"},
    {"amount_eur": 50.0, "original_amount": 50.0, "original_currency": "EUR", "category": "Transport", "description": "Train"},
    {"amount_eur": 42.0, "original_amount": 35.0, "original_currency": "GBP", "category": "Entertainment", "description": "London show"},
]

def multi_currency_report(transactions):
    groups = {}
    for t in transactions:
        curr = t["original_currency"]
        if curr not in groups:
            groups[curr] = {"count": 0, "original": 0, "eur": 0}
        groups[curr]["count"] += 1
        groups[curr]["original"] += t["original_amount"]
        groups[curr]["eur"] += t["amount_eur"]

    # only add up the eur values, cant add usd + gbp together
    total_eur = 0
    for curr in groups:
        total_eur += groups[curr]["eur"]

    print("--- Multi Currency Report ---")
    for curr, info in groups.items():
        print(f"{curr}: {info['count']} transaction(s), {info['original']} {curr} = {info['eur']} EUR")
    print("Total in EUR:", total_eur)
    return groups

multi_currency_report(transactions)

--- Multi Currency Report ---
USD: 1 transaction(s), 27.5 USD = 25.0 EUR
EUR: 1 transaction(s), 50.0 EUR = 50.0 EUR
GBP: 1 transaction(s), 35.0 GBP = 42.0 EUR
Total in EUR: 117.0


{'USD': {'count': 1, 'original': 27.5, 'eur': 25.0},
 'EUR': {'count': 1, 'original': 50.0, 'eur': 50.0},
 'GBP': {'count': 1, 'original': 35.0, 'eur': 42.0}}

---
## Exercise 6: Cached Currency Lookup
**Objective:** Build a resilient API fetch that caches results and handles failures cleanly.

Write a function `get_exchange_rates(base_currency="USD", cache_file="rates_cache.json")` that:
- fetches `https://open.er-api.com/v6/latest/{base_currency}` with `requests.get()` and a timeout
- handles `Timeout`, `HTTPError`, and general `RequestException`
- treats a response whose `result` is not `"success"` as a failure
- returns cached data when the file is fresh **and** holds rates for the requested `base_currency` (its `base_code`)
- saves the latest JSON response when the fetch succeeds

This exercise simulates the kind of API code used in real dashboards and automation tasks.

### How to approach this code

This version combines cache freshness, JSON read/write, a request timeout, and failure handling. It should return the same data shape whether the result came from cache or network, and write the cache only after a successful response. These choices keep downstream code predictable during outages.


In [32]:
import json
import os
import time
import requests

def get_exchange_rates(base_currency="USD", cache_file="rates_cache.json", max_age=3600):
    base_currency = base_currency.upper()

    # use the saved file if its still fresh and its for the same currency
    if os.path.exists(cache_file):
        age = time.time() - os.path.getmtime(cache_file)
        if age < max_age:
            try:
                with open(cache_file, encoding="utf-8") as f:
                    cached = json.load(f)
                if cached.get("base_code") == base_currency:
                    print("using cache for", base_currency)
                    return cached
            except json.JSONDecodeError:
                print("cache file is broken, fetching again")

    url = f"https://open.er-api.com/v6/latest/{base_currency}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.Timeout:
        print("request timed out")
        return {}
    except requests.exceptions.HTTPError as e:
        print("http error:", e.response.status_code)
        return {}
    except requests.exceptions.RequestException as e:
        print("request failed:", e)
        return {}

    # api still gives 200 for a wrong code, so check result as well
    if data.get("result") != "success":
        print("api error:", data.get("error-type"))
        return {}

    # only write the file here so a failed call doesnt wipe the old cache
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    print("fetched new rates for", base_currency)
    return data


usd = get_exchange_rates("USD")
print(usd.get("rates", {}).get("EUR"))

usd = get_exchange_rates("USD")  # should come from the cache now
eur = get_exchange_rates("EUR")  # different currency, cant use the USD cache
print(eur.get("base_code"))

print(get_exchange_rates("INVALID"))

fetched new rates for USD
0.9121
using cache for USD
fetched new rates for EUR
EUR
api error: unsupported-code
{}
